# 📊 Event Market Scanner — Notebook

Download prediction market data (top 500 events by liquidity) from **Polymarket** and **Kalshi**, explore it in DataFrames, and export to CSV/Excel/Parquet.

## 1. Import Required Libraries

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

# Import collectors from the scanner module
from event_market_scanner import PolymarketCollector, KalshiCollector, Event

print("Libraries loaded ✓")

## 2. Configuration

Set the number of events to fetch per platform and the output directory.

In [ ]:
# Number of top events to fetch per platform (max 1000)
LIMIT = 1000

# Output directory for exported files
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Will fetch top {LIMIT} events per platform")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

## 3. Fetch Data from Polymarket & Kalshi

Download the top events sorted by liquidity from both platforms.

In [ ]:
# Fetch Polymarket events
print("⏳ Fetching Polymarket events...")
poly_collector = PolymarketCollector()
poly_events = poly_collector.collect(limit=LIMIT)
print(f"✅ Polymarket: {len(poly_events)} events collected")

# Fetch Kalshi events
print("\n⏳ Fetching Kalshi events...")
kalshi_collector = KalshiCollector()
kalshi_events = kalshi_collector.collect(limit=LIMIT)
print(f"✅ Kalshi: {len(kalshi_events)} events collected")

print(f"\n📊 Total: {len(poly_events) + len(kalshi_events)} events")

## 4. Build DataFrames

Convert the raw event data into flat DataFrames — one row per outcome — for easy exploration and export.

In [ ]:
def events_to_rows(events, source_label=None):
    """Flatten events → list of dicts (one row per outcome)."""
    rows = []
    for rank, event in enumerate(events, 1):
        for market in event.markets:
            for outcome in market.outcomes:
                rows.append({
                    "Rank": rank,
                    "Source": source_label or event.source,
                    "Event": event.title,
                    "Event Liquidity ($)": round(event.liquidity, 2),
                    "Event Volume ($)": round(event.volume, 2),
                    "Market": market.question,
                    "Market Liquidity ($)": round(market.liquidity, 2),
                    "Outcome": outcome.name,
                    "Price": round(outcome.price, 4),
                    "Implied Prob (%)": round(outcome.price * 100, 2),
                    "URL": event.url,
                    "Category": event.category,
                })
    return rows

# Build separate DataFrames
df_poly = pd.DataFrame(events_to_rows(poly_events, "Polymarket"))
df_kalshi = pd.DataFrame(events_to_rows(kalshi_events, "Kalshi"))

# Combined DataFrame
df_all = pd.concat([df_poly, df_kalshi], ignore_index=True)

print(f"Polymarket rows: {len(df_poly):,}")
print(f"Kalshi rows:     {len(df_kalshi):,}")
print(f"Combined rows:   {len(df_all):,}")

## 5. Explore the Data

### Polymarket — Top Events

In [ ]:
# Summary table: one row per event (deduplicated from the per-outcome rows)
poly_summary = (
    df_poly.groupby(["Rank", "Event"])
    .agg({"Event Liquidity ($)": "first", "Event Volume ($)": "first", "Market": "nunique", "Outcome": "count"})
    .rename(columns={"Market": "Markets", "Outcome": "Outcomes"})
    .sort_values("Rank")
    .head(25)
)
poly_summary.style.format({
    "Event Liquidity ($)": "${:,.0f}",
    "Event Volume ($)": "${:,.0f}",
})

### Kalshi — Top Events

In [ ]:
kalshi_summary = (
    df_kalshi.groupby(["Rank", "Event"])
    .agg({"Event Liquidity ($)": "first", "Event Volume ($)": "first", "Market": "nunique", "Outcome": "count"})
    .rename(columns={"Market": "Markets", "Outcome": "Outcomes"})
    .sort_values("Rank")
    .head(25)
)
kalshi_summary.style.format({
    "Event Liquidity ($)": "${:,.0f}",
    "Event Volume ($)": "${:,.0f}",
})

### Data Overview & Statistics

In [ ]:
print("=== Combined Dataset Shape ===")
print(f"Rows: {len(df_all):,}  |  Columns: {len(df_all.columns)}")
print(f"\n=== Columns ===")
print(df_all.dtypes.to_string())
print(f"\n=== Numeric Summary ===")
df_all[["Event Liquidity ($)", "Event Volume ($)", "Market Liquidity ($)", "Price", "Implied Prob (%)"]].describe()

## 6. Detailed Event Viewer

Shows the top events from each platform with all outcome prices. Change `NUM_EVENTS` below to control how many are displayed.

In [ ]:
from IPython.display import display, HTML

def show_event_detail(event, rank, source):
    """Render a rich HTML card for a single event with all outcome prices.
    Combines binary Yes/No markets into a single consolidated view.
    Skips markets that have no outcomes."""

    # Filter out markets with no outcomes
    valid_markets = [m for m in event.markets if m.outcomes]
    if not valid_markets:
        return  # nothing to display

    binary_markets = [
        m for m in valid_markets
        if len(m.outcomes) == 2
        and {o.name for o in m.outcomes} == {"Yes", "No"}
    ]
    non_binary_markets = [m for m in valid_markets if m not in binary_markets]

    display_groups = []

    # If most markets are binary, consolidate them into one table
    if len(binary_markets) >= 3:
        consolidated_outcomes = []
        total_liq = 0.0
        for m in binary_markets:
            yes_price = next((o.price for o in m.outcomes if o.name == "Yes"), 0)
            label = m.question
            for prefix in [event.title + " ", "Will ", "Will the "]:
                if label.startswith(prefix):
                    label = label[len(prefix):]
                    break
            label = label.rstrip("?").strip()
            if label:
                label = label[0].upper() + label[1:]
            if not label:
                continue  # skip outcomes with no label
            consolidated_outcomes.append((label, yes_price, m.liquidity))
            total_liq += m.liquidity

        if consolidated_outcomes:
            consolidated_outcomes.sort(key=lambda x: x[1], reverse=True)
            display_groups.append(("consolidated", event.title, consolidated_outcomes, total_liq))
    else:
        for m in binary_markets:
            outcomes = [(o.name, o.price, m.liquidity) for o in sorted(m.outcomes, key=lambda o: o.price, reverse=True)]
            display_groups.append(("market", m.question, outcomes, m.liquidity))

    for m in non_binary_markets:
        outcomes = [(o.name, o.price, m.liquidity) for o in sorted(m.outcomes, key=lambda o: o.price, reverse=True) if o.name]
        if outcomes:
            display_groups.append(("market", m.question, outcomes, m.liquidity))

    if not display_groups:
        return  # nothing to display after filtering

    # ── Build HTML ──
    html = f"""
    <div style="border:2px solid #444; border-radius:12px; padding:20px; margin:16px 0;
                background:linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); color:#e0e0e0;
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <h2 style="margin:0 0 4px 0; color:#00d4ff;">#{rank} — {event.title}</h2>
        <p style="margin:0 0 12px 0; color:#888; font-size:13px;">
            <b>{source}</b> &nbsp;|&nbsp;
            Liquidity: <span style="color:#4caf50; font-weight:600;">${event.liquidity:,.0f}</span> &nbsp;|&nbsp;
            Volume: <span style="color:#2196f3; font-weight:600;">${event.volume:,.0f}</span> &nbsp;|&nbsp;
            Markets: <b>{len(valid_markets)}</b>
            {"&nbsp;|&nbsp; <a href='" + event.url + "' style='color:#64b5f6;'>View on " + source + " ↗</a>" if event.url else ""}
        </p>
    """

    for group_type, label, outcomes, liq in display_groups:
        tag = "All Outcomes (consolidated)" if group_type == "consolidated" else ""
        liq_str = f"&nbsp;&nbsp;<span style='color:#666; font-size:11px;'>Liquidity: ${liq:,.0f}</span>"
        html += f"""
        <div style="margin:12px 0 6px 0; padding:10px 14px; background:#0d1b30; border-radius:8px;
                    border-left:4px solid #00d4ff;">
            <p style="margin:0 0 2px 0; font-weight:600; font-size:14px; color:#b0bec5;">{label}</p>
            <p style="margin:0 0 8px 0; font-size:11px; color:#555;">{tag}{liq_str}</p>
            <table style="width:100%; border-collapse:collapse; font-size:13px;">
                <tr style="border-bottom:1px solid #333;">
                    <th style="text-align:left; padding:4px 8px; color:#888; width:45%;">Outcome</th>
                    <th style="text-align:right; padding:4px 8px; color:#888; width:18%;">Price</th>
                    <th style="text-align:right; padding:4px 8px; color:#888; width:22%;">Probability</th>
                    <th style="text-align:left; padding:4px 8px; color:#888; width:15%;"></th>
                </tr>
        """
        for name, price, _ in outcomes:
            pct = price * 100
            if pct >= 50:
                bar_color = "#4caf50"
            elif pct >= 15:
                bar_color = "#ff9800"
            else:
                bar_color = "#f44336"
            bar_width = max(pct, 1)
            html += f"""
                <tr style="border-bottom:1px solid #222;">
                    <td style="padding:5px 8px; color:#e0e0e0; font-weight:500;">{name}</td>
                    <td style="padding:5px 8px; text-align:right; color:#e0e0e0; font-family:monospace;">${price:.4f}</td>
                    <td style="padding:5px 8px; text-align:right; color:{bar_color}; font-weight:600; font-family:monospace;">{pct:.1f}%</td>
                    <td style="padding:5px 8px;">
                        <div style="background:#1a1a2e; border-radius:4px; height:14px; width:100%;">
                            <div style="background:{bar_color}; border-radius:4px; height:14px; width:{bar_width}%;"></div>
                        </div>
                    </td>
                </tr>
            """
        html += "</table></div>"

    html += "</div>"
    display(HTML(html))


# ───── Change this to control how many events are shown ─────
NUM_EVENTS = 10  # Show top N events from each platform

# ── Polymarket ──
if poly_events:
    display(HTML("<h2 style='color:#00d4ff; border-bottom:2px solid #00d4ff; padding-bottom:6px;'>🔮 Polymarket — Top Events</h2>"))
    for i in range(min(NUM_EVENTS, len(poly_events))):
        show_event_detail(poly_events[i], i + 1, "Polymarket")

# ── Kalshi ──
if kalshi_events:
    display(HTML("<h2 style='color:#00d4ff; border-bottom:2px solid #00d4ff; padding-bottom:6px;'>📊 Kalshi — Top Events</h2>"))
    for i in range(min(NUM_EVENTS, len(kalshi_events))):
        show_event_detail(kalshi_events[i], i + 1, "Kalshi")

## 7. Cross-Platform Arbitrage Scanner

Matches events and outcomes across Polymarket and Kalshi using **semantic-aware** fuzzy matching, then identifies pricing discrepancies.

### Matching Intelligence
The matcher understands that these are the same outcome:
- **Dates**: "before April 1st" ↔ "by March 31st" ↔ "by end of March"
- **Numbers**: "$50K" ↔ "$50,000" ↔ "50000", ordinals like "1st"/"first"
- **Phrasing**: "more than" ↔ "over" ↔ "above" ↔ "exceeding" ↔ "at least"
- **Directions**: "increase" ↔ "rise" ↔ "go up", "decrease" ↔ "fall" ↔ "drop"
- **Names**: "Donald Trump" ↔ "Trump", "J.D. Vance" ↔ "JD Vance"
- **Negation**: Detects inverted markets (Yes/No flipped) and adjusts prices accordingly

### Arbitrage Detection — All Price Comparisons
Both strategies are checked using **actual Yes and No prices** from each platform (never assuming they sum to 100%):

| Strategy | You Buy | Cost | Profit |
|---|---|---|---|
| **A** | Yes on Poly + No on Kalshi | $P_Y + K_N$ | $1 - (P_Y + K_N)$ |
| **B** | Yes on Kalshi + No on Poly | $K_Y + P_N$ | $1 - (K_Y + P_N)$ |

A **true arbitrage** exists when the best strategy costs **< $1.00**. Platform overround (Yes + No > 1.00) is shown to highlight vig.

**Configuration:**
- `MATCH_THRESHOLD` — minimum similarity score (0–1) for matching
- `MIN_SPREAD` — minimum price difference to flag
- `SEMANTIC_BONUS` — extra score for semantic equivalence detected

In [ ]:
import re, calendar
from difflib import SequenceMatcher
from datetime import date, timedelta
from IPython.display import display, HTML

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MATCH_THRESHOLD  = 0.50   # Min combined score to consider a match
MIN_SPREAD       = 0.01   # Min price spread ($) to display
TOP_ARBS         = 30     # Max rows in summary tables
SEMANTIC_BONUS   = 0.15   # Extra score when semantic normalization detects equivalence
TOKEN_WEIGHT     = 0.35   # Weight of token-overlap score vs fuzzy string score

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  1. DATE NORMALISATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

_MONTH_MAP = {m.lower(): i for i, m in enumerate(calendar.month_name) if m}
_MONTH_MAP.update({m.lower(): i for i, m in enumerate(calendar.month_abbr) if m})

_ORDINAL_RE = re.compile(r"(\d+)\s*(?:st|nd|rd|th)\b", re.I)
_MONTH_NAMES = "|".join(list(calendar.month_name[1:]) + list(calendar.month_abbr[1:]))

# "before April 1" / "before April 1st" / "before April 1, 2026"
_BEFORE_DATE_RE = re.compile(
    rf"\bbefore\s+({_MONTH_NAMES})\s+(\d{{1,2}})(?:\s*(?:st|nd|rd|th))?(?:\s*,?\s*(\d{{4}}))?",
    re.I
)
# "by March 31" / "by March 31st"
_BY_DATE_RE = re.compile(
    rf"\bby\s+({_MONTH_NAMES})\s+(\d{{1,2}})(?:\s*(?:st|nd|rd|th))?(?:\s*,?\s*(\d{{4}}))?",
    re.I
)
# "by end of March" / "by the end of Q1"
_BY_END_OF_RE = re.compile(
    rf"\bby\s+(?:the\s+)?end\s+of\s+({_MONTH_NAMES})\s*,?\s*(\d{{4}})?",
    re.I
)
# Quarter references
_QUARTER_ENDS = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
_BY_END_Q_RE = re.compile(r"\bby\s+(?:the\s+)?end\s+of\s+q([1-4])\s*,?\s*(\d{4})?", re.I)

def _last_day(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]

def _canon_date(month_str: str, day: int, year: int | None) -> str:
    """Convert to canonical 'by YYYY-MM-DD' string."""
    m = _MONTH_MAP.get(month_str.lower())
    if m is None:
        return ""
    y = year or 2026  # default to current year context
    d = min(day, _last_day(y, m))
    return f"by {y:04d}-{m:02d}-{d:02d}"

def _normalise_dates(text: str) -> str:
    """Convert date phrases to canonical form so equivalences match."""

    # "before April 1" → "by March 31" (subtract one day)
    def _before_repl(m):
        mon, day, yr = m.group(1), int(m.group(2)), m.group(3)
        mi = _MONTH_MAP.get(mon.lower())
        if mi is None:
            return m.group(0)
        y = int(yr) if yr else 2026
        d = date(y, mi, min(int(day), _last_day(y, mi)))
        prev = d - timedelta(days=1)
        return f"by {prev.year:04d}-{prev.month:02d}-{prev.day:02d}"

    text = _BEFORE_DATE_RE.sub(_before_repl, text)

    # "by March 31" → canonical
    def _by_repl(m):
        return _canon_date(m.group(1), int(m.group(2)), int(m.group(3)) if m.group(3) else None)
    text = _BY_DATE_RE.sub(_by_repl, text)

    # "by end of March" → "by YYYY-03-31"
    def _by_end_repl(m):
        mon, yr = m.group(1), m.group(2)
        mi = _MONTH_MAP.get(mon.lower())
        if mi is None:
            return m.group(0)
        y = int(yr) if yr else 2026
        return f"by {y:04d}-{mi:02d}-{_last_day(y, mi):02d}"
    text = _BY_END_OF_RE.sub(_by_end_repl, text)

    # "by end of Q1" → "by YYYY-03-31"
    def _q_repl(m):
        q, yr = int(m.group(1)), m.group(2)
        y = int(yr) if yr else 2026
        em, ed = _QUARTER_ENDS[q]
        return f"by {y:04d}-{em:02d}-{ed:02d}"
    text = _BY_END_Q_RE.sub(_q_repl, text)

    return text

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  2. NUMBER & UNIT NORMALISATION                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

_ORDINAL_WORDS = {
    "first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5",
    "sixth": "6", "seventh": "7", "eighth": "8", "ninth": "9", "tenth": "10",
}

def _normalise_numbers(text: str) -> str:
    """Standardise numeric expressions."""
    # Ordinal words → digits
    for word, digit in _ORDINAL_WORDS.items():
        text = re.sub(rf"\b{word}\b", digit, text, flags=re.I)
    # Strip ordinal suffixes: "1st" → "1"
    text = _ORDINAL_RE.sub(r"\1", text)
    # "$50K" / "$50k" → "$50000"
    text = re.sub(r"\$\s*([\d,.]+)\s*[kK]\b", lambda m: "$" + str(int(float(m.group(1).replace(",", "")) * 1000)), text)
    # "$1.5M" / "$1.5m" → "$1500000"
    text = re.sub(r"\$\s*([\d,.]+)\s*[mM]\b", lambda m: "$" + str(int(float(m.group(1).replace(",", "")) * 1_000_000)), text)
    # "$1.2B" / "$1.2b" → "$1200000000"
    text = re.sub(r"\$\s*([\d,.]+)\s*[bB]\b", lambda m: "$" + str(int(float(m.group(1).replace(",", "")) * 1_000_000_000)), text)
    # Strip commas in numbers: "50,000" → "50000"
    text = re.sub(r"(\d),(\d{3})", r"\1\2", text)
    text = re.sub(r"(\d),(\d{3})", r"\1\2", text)  # double pass for millions
    # "percent" / "pct" / "%" normalisation
    text = re.sub(r"\bpercent\b", "%", text, flags=re.I)
    text = re.sub(r"\bpct\b", "%", text, flags=re.I)
    return text

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  3. SYNONYM / PHRASING NORMALISATION                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Directional / threshold synonyms → canonical form
_SYNONYM_GROUPS = [
    # Comparison — "more than" family
    (["more than", "greater than", "over", "above", "exceeding", "exceed",
      "higher than", "in excess of"], "more than"),
    # Comparison — "less than" family
    (["less than", "fewer than", "under", "below", "lower than"], "less than"),
    # Comparison — "at least" family
    (["at least", "no fewer than", "no less than", "a minimum of", "or more"], "at least"),
    # Comparison — "at most" family
    (["at most", "no more than", "a maximum of", "or fewer", "or less"], "at most"),
    # Direction — up
    (["increase", "rise", "go up", "gain", "climb", "grow", "surge", "jump"], "increase"),
    # Direction — down
    (["decrease", "fall", "go down", "decline", "drop", "sink", "plunge", "slide"], "decrease"),
    # Win/victory
    (["win", "defeat", "beat", "prevail"], "win"),
    # Confirmation
    (["confirm", "approve", "ratify", "pass"], "confirm"),
    # Announce / reveal
    (["announce", "reveal", "disclose", "unveil"], "announce"),
]

def _normalise_synonyms(text: str) -> str:
    """Replace synonym variants with a canonical form."""
    for synonyms, canonical in _SYNONYM_GROUPS:
        for syn in synonyms:
            if syn == canonical:
                continue
            text = re.sub(rf"\b{re.escape(syn)}\b", canonical, text, flags=re.I)
    return text

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  4. NAME NORMALISATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

_NAME_VARIANTS = {
    # US political figures (extend as needed)
    "donald trump": "trump", "donald j trump": "trump", "president trump": "trump",
    "joe biden": "biden", "joseph biden": "biden", "president biden": "biden",
    "kamala harris": "harris", "vice president harris": "harris",
    "j d vance": "jd vance", "j.d. vance": "jd vance",
    "ron desantis": "desantis", "ronald desantis": "desantis",
    "elon musk": "musk",
    "jerome powell": "powell", "jay powell": "powell", "chair powell": "powell",
    "jerome h powell": "powell",
    # Institutions
    "federal reserve": "fed", "the fed": "fed",
    "european central bank": "ecb",
    "united states": "us", "united states of america": "us",
    "united kingdom": "uk",
    "s&p 500": "sp500", "s&p500": "sp500", "s and p 500": "sp500",
    "bitcoin": "btc", "ethereum": "eth",
}

def _normalise_names(text: str) -> str:
    """Replace known name variants with canonical short forms."""
    for variant, canon in sorted(_NAME_VARIANTS.items(), key=lambda x: -len(x[0])):
        text = re.sub(rf"\b{re.escape(variant)}\b", canon, text, flags=re.I)
    return text

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  5. NEGATION DETECTION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

_NEGATION_PATTERNS = [
    re.compile(r"\bwill\s+not\b", re.I),
    re.compile(r"\bwon't\b", re.I),
    re.compile(r"\bnot\s+(?:be|happen|occur|pass|win)\b", re.I),
    re.compile(r"\bfail(?:s)?\s+to\b", re.I),
    re.compile(r"\bno\s+(?:new|further|additional)\b", re.I),
]

def _has_negation(text: str) -> bool:
    """Detect if the text contains a negation."""
    return any(p.search(text) for p in _NEGATION_PATTERNS)

def _strip_negation(text: str) -> str:
    """Remove negation words for comparison purposes."""
    text = re.sub(r"\bwill\s+not\b", "will", text, flags=re.I)
    text = re.sub(r"\bwon't\b", "will", text, flags=re.I)
    text = re.sub(r"\bnot\s+", " ", text, flags=re.I)
    text = re.sub(r"\bfail(?:s)?\s+to\b", "", text, flags=re.I)
    return text

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  6. COMBINED NORMALISATION PIPELINE                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _normalise(text: str) -> str:
    """Full normalisation pipeline: dates → numbers → synonyms → names → cleanup."""
    text = text.lower().strip()
    text = _normalise_dates(text)
    text = _normalise_numbers(text)
    text = _normalise_synonyms(text)
    text = _normalise_names(text)
    # Strip remaining punctuation and collapse whitespace
    text = re.sub(r"""['''"?!.,;:()\[\]{}&]""", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _extract_key_tokens(text: str) -> set[str]:
    """Extract salient tokens: numbers, dates, $-amounts, and longer words."""
    normed = _normalise(text)
    tokens = set()
    # Dollar amounts
    tokens.update(re.findall(r"\$\d+", normed))
    # Standalone numbers (including year-like)
    tokens.update(re.findall(r"\b\d{2,}\b", normed))
    # Date patterns already canonicalised
    tokens.update(re.findall(r"\d{4}-\d{2}-\d{2}", normed))
    # Significant words (4+ chars, not stopwords)
    _STOP = {"will", "what", "when", "where", "which", "that", "this", "with", "from",
             "have", "been", "does", "than", "more", "less", "into", "also", "each",
             "they", "them", "then", "some", "most", "about", "their", "there", "would",
             "could", "should", "being", "other", "after", "before", "between"}
    for w in normed.split():
        if len(w) >= 4 and w not in _STOP and not w.startswith("$"):
            tokens.add(w)
    return tokens

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  7. MULTI-SIGNAL SIMILARITY SCORING                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _combined_score(a: str, b: str) -> tuple[float, bool]:
    """Return (similarity 0-1, is_negation_inverted).

    Combines:
      - SequenceMatcher ratio on normalised strings
      - Jaccard token overlap on key tokens
      - Semantic bonus if normalisation made strings converge
      - Negation detection for price inversion
    """
    raw_a, raw_b = a, b
    na, nb = _normalise(a), _normalise(b)

    # Fuzzy string similarity on normalised text
    fuzzy = SequenceMatcher(None, na, nb).ratio()

    # Token overlap (Jaccard)
    ta, tb = _extract_key_tokens(a), _extract_key_tokens(b)
    if ta and tb:
        jaccard = len(ta & tb) / len(ta | tb)
    else:
        jaccard = 0.0

    # Blend fuzzy + token scores
    score = (1.0 - TOKEN_WEIGHT) * fuzzy + TOKEN_WEIGHT * jaccard

    # Semantic bonus: if normalisation improved the match compared to raw
    raw_fuzzy = SequenceMatcher(None, raw_a.lower(), raw_b.lower()).ratio()
    if fuzzy > raw_fuzzy + 0.05:
        score = min(1.0, score + SEMANTIC_BONUS)

    # Negation handling: if one has negation and the other doesn't
    neg_a = _has_negation(raw_a)
    neg_b = _has_negation(raw_b)
    inverted = neg_a != neg_b

    if inverted:
        # Try matching WITHOUT the negation
        stripped_a = _strip_negation(na)
        stripped_b = _strip_negation(nb)
        stripped_fuzzy = SequenceMatcher(None, stripped_a, stripped_b).ratio()
        if stripped_fuzzy > fuzzy:
            score = (1.0 - TOKEN_WEIGHT) * stripped_fuzzy + TOKEN_WEIGHT * jaccard
            if stripped_fuzzy > raw_fuzzy + 0.05:
                score = min(1.0, score + SEMANTIC_BONUS)

    return score, inverted

def _best_match(needle: str, haystack: list[str], threshold: float) -> tuple[int, float, bool] | None:
    """Find best matching string. Returns (index, score, is_inverted) or None."""
    best_idx, best_score, best_inv = -1, 0.0, False
    for i, h in enumerate(haystack):
        score, inv = _combined_score(needle, h)
        if score > best_score:
            best_score = score
            best_idx = i
            best_inv = inv
    if best_score >= threshold:
        return best_idx, best_score, best_inv
    return None

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  8. OUTCOME EXTRACTION                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _extract_outcomes(event) -> dict[str, dict]:
    """Flatten event → {normalised_label: {label, yes_price, no_price, ...}}
    Uses ACTUAL Yes and No prices from the market — never assumes they sum to 1.00.
    """
    outcomes = {}
    for m in event.markets:
        if not m.outcomes:
            continue
        outcome_names = {o.name for o in m.outcomes}

        if len(m.outcomes) == 2 and outcome_names == {"Yes", "No"}:
            yes_p = next(o.price for o in m.outcomes if o.name == "Yes")
            no_p  = next(o.price for o in m.outcomes if o.name == "No")
            label = m.question.strip()
            for prefix in [event.title + " ", "Will ", "Will the "]:
                if label.startswith(prefix):
                    label = label[len(prefix):]
                    break
            label = label.rstrip("?").strip()
            if label:
                label = label[0].upper() + label[1:]
            else:
                label = m.question
            key = _normalise(label)
            outcomes[key] = {
                "label": label,
                "yes_price": round(yes_p, 4),
                "no_price": round(no_p, 4),
                "overround": round(yes_p + no_p, 4),
                "market": m.question,
                "liquidity": m.liquidity,
            }
        else:
            for o in m.outcomes:
                if not o.name:
                    continue
                key = _normalise(o.name)
                # For multi-outcome markets we only have one price per outcome;
                # the complement is implied but may not be tradeable, so store
                # the raw price as yes_price and 1-price as no_price (best we can do).
                outcomes[key] = {
                    "label": o.name,
                    "yes_price": round(o.price, 4),
                    "no_price": round(1.0 - o.price, 4),
                    "overround": 1.0,  # unknown for multi-outcome
                    "market": m.question,
                    "liquidity": m.liquidity,
                }
    return outcomes

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  9. MATCH EVENTS ACROSS PLATFORMS                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

matched_events = []
used_kalshi = set()

print("🔍 Matching events across platforms (semantic-aware)...")
for pi, pe in enumerate(poly_events):
    best_ki, best_score, best_inv = -1, 0.0, False
    for ki, ke in enumerate(kalshi_events):
        if ki in used_kalshi:
            continue
        score, inv = _combined_score(pe.title, ke.title)
        if score > best_score:
            best_score = score
            best_ki = ki
            best_inv = inv
    if best_score >= MATCH_THRESHOLD and best_ki >= 0:
        matched_events.append((pe, kalshi_events[best_ki], best_score))
        used_kalshi.add(best_ki)

print(f"✅ Matched {len(matched_events)} events across Polymarket ↔ Kalshi\n")

# Show some top matches for verification
if matched_events:
    print("  Top event matches:")
    for pe, ke, sim in sorted(matched_events, key=lambda x: x[2], reverse=True)[:8]:
        emoji = "🟢" if sim >= 0.85 else ("🟡" if sim >= 0.7 else "🟠")
        print(f"  {emoji} {sim:.0%}  {pe.title[:45]:45s} ↔ {ke.title[:45]}")
    print()

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  10. MATCH OUTCOMES & DETECT ARBITRAGE                                   ║
# ║                                                                          ║
# ║  Checks BOTH cross-platform strategies using actual Yes & No prices:     ║
# ║    Strategy A: Buy Yes @ Poly  + Buy No  @ Kalshi → cost_a              ║
# ║    Strategy B: Buy Yes @ Kalshi + Buy No  @ Poly  → cost_b              ║
# ║  Also compares Yes-vs-Yes and No-vs-No for spread visibility.            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

arb_rows = []

for pe, ke, event_sim in matched_events:
    poly_outs  = _extract_outcomes(pe)
    kalshi_outs = _extract_outcomes(ke)

    if not poly_outs or not kalshi_outs:
        continue

    kalshi_keys = list(kalshi_outs.keys())
    kalshi_labels = [kalshi_outs[k]["label"] for k in kalshi_keys]
    used_k = set()

    for pk, pdata in poly_outs.items():
        # Try exact normalised match first
        if pk in kalshi_outs and pk not in used_k:
            kdata = kalshi_outs[pk]
            match_score = 1.0
            is_inverted = False
            used_k.add(pk)
        else:
            result = _best_match(pdata["label"], kalshi_labels, MATCH_THRESHOLD)
            if result is None:
                continue
            idx, match_score, is_inverted = result
            kk = kalshi_keys[idx]
            if kk in used_k:
                continue
            kdata = kalshi_outs[kk]
            used_k.add(kk)

        # ── Gather all 4 raw prices ──
        poly_yes = pdata["yes_price"]
        poly_no  = pdata["no_price"]

        if is_inverted:
            # Negation-inverted: Kalshi's Yes ↔ No are flipped relative to Poly
            kalshi_yes = kdata["no_price"]
            kalshi_no  = kdata["yes_price"]
        else:
            kalshi_yes = kdata["yes_price"]
            kalshi_no  = kdata["no_price"]

        # ── Platform overround (vig) ──
        poly_overround  = pdata["overround"]
        kalshi_overround = kdata["overround"]

        # ── Both cross-platform strategies ──
        cost_a = poly_yes + kalshi_no    # Strategy A: Yes@Poly + No@Kalshi
        cost_b = kalshi_yes + poly_no    # Strategy B: Yes@Kalshi + No@Poly
        profit_a = max(0, 1.0 - cost_a)
        profit_b = max(0, 1.0 - cost_b)

        # Pick the better strategy
        if profit_a >= profit_b:
            best_cost   = cost_a
            best_profit = profit_a
            best_strategy = "A"
            direction = "Buy Yes → Poly, Buy No → Kalshi"
        else:
            best_cost   = cost_b
            best_profit = profit_b
            best_strategy = "B"
            direction = "Buy Yes → Kalshi, Buy No → Poly"

        if best_profit == 0 and cost_a == cost_b:
            direction = "No edge"

        if is_inverted:
            direction += " ⚠️ inverted"

        # ── Spreads on each side ──
        yes_spread = abs(poly_yes - kalshi_yes)
        no_spread  = abs(poly_no - kalshi_no)
        max_spread = max(yes_spread, no_spread)

        if max_spread >= MIN_SPREAD:
            arb_rows.append({
                "event": pe.title,
                "event_sim": event_sim,
                "outcome": pdata["label"],
                "outcome_kalshi": kdata["label"],
                "outcome_sim": match_score,
                "inverted": is_inverted,
                # All 4 prices
                "poly_yes": poly_yes,
                "poly_no": poly_no,
                "kalshi_yes": kalshi_yes,
                "kalshi_no": kalshi_no,
                # Platform overround
                "poly_overround": poly_overround,
                "kalshi_overround": kalshi_overround,
                # Side spreads
                "yes_spread": yes_spread,
                "no_spread": no_spread,
                # Both strategies
                "cost_a": cost_a,
                "cost_b": cost_b,
                "profit_a": profit_a,
                "profit_b": profit_b,
                # Best
                "best_strategy": best_strategy,
                "best_cost": best_cost,
                "arb_profit": best_profit,
                "direction": direction,
                # Liquidity
                "poly_liq": pdata["liquidity"],
                "kalshi_liq": kdata["liquidity"],
                "poly_url": pe.url,
                "kalshi_url": ke.url,
            })

arb_rows.sort(key=lambda r: (r["arb_profit"], max(r["yes_spread"], r["no_spread"])), reverse=True)

print(f"📊 Found {len(arb_rows)} outcome pairs with spread ≥ ${MIN_SPREAD:.2f}")
true_arbs = [r for r in arb_rows if r["arb_profit"] > 0]
print(f"🎯 True arbitrage opportunities (profit > 0): {len(true_arbs)}")
inverted_count = sum(1 for r in arb_rows if r["inverted"])
if inverted_count:
    print(f"🔄 Negation-inverted matches detected: {inverted_count}")

# Overround stats
overrounds = [(r["poly_overround"], r["kalshi_overround"]) for r in arb_rows if r["poly_overround"] != 1.0]
if overrounds:
    avg_poly_or  = sum(p for p, _ in overrounds) / len(overrounds)
    avg_kalshi_or = sum(k for _, k in overrounds) / len(overrounds)
    print(f"📐 Avg overround — Poly: {avg_poly_or:.1%}  Kalshi: {avg_kalshi_or:.1%}")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  11. DISPLAY                                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

if not arb_rows:
    display(HTML("""
    <div style="padding:24px; text-align:center; color:#888; font-size:16px;
                border:2px dashed #444; border-radius:12px; margin:16px 0;">
        No matching outcomes found between platforms at the current threshold.<br>
        Try lowering <code>MATCH_THRESHOLD</code>.
    </div>"""))
else:
    # ── True Arbitrage Table ──
    if true_arbs:
        arb_html = """
        <div style="border:2px solid #4caf50; border-radius:12px; padding:20px; margin:16px 0;
                    background:linear-gradient(135deg, #0a2e0a 0%, #1a3a1a 100%); color:#e0e0e0;
                    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
            <h2 style="margin:0 0 8px 0; color:#4caf50;">🎯 True Arbitrage Opportunities</h2>
            <p style="margin:0 0 14px 0; color:#888; font-size:13px;">
                Total cost &lt; $1.00 across platforms — guaranteed profit if both legs execute.
                All 4 prices shown (platform odds may not sum to 100%).
            </p>
            <table style="width:100%; border-collapse:collapse; font-size:12px;">
                <tr style="border-bottom:2px solid #4caf50;">
                    <th style="text-align:left; padding:6px 6px; color:#4caf50;">Event</th>
                    <th style="text-align:left; padding:6px 6px; color:#4caf50;">Outcome</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">P.Yes</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">P.No</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">K.Yes</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">K.No</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">Cost A</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">Cost B</th>
                    <th style="text-align:right; padding:6px 6px; color:#4caf50;">Profit</th>
                    <th style="text-align:left; padding:6px 6px; color:#4caf50;">Best</th>
                    <th style="text-align:center; padding:6px 6px; color:#4caf50;">Match</th>
                </tr>
        """
        for r in true_arbs[:TOP_ARBS]:
            profit_pct = r["arb_profit"] * 100
            match_emoji = "🟢" if r["outcome_sim"] >= 0.85 else ("🟡" if r["outcome_sim"] >= 0.7 else "🟠")
            inv_badge = " 🔄" if r["inverted"] else ""
            # Highlight the winning cost
            cost_a_style = "color:#4caf50; font-weight:700;" if r["best_strategy"] == "A" else "color:#888;"
            cost_b_style = "color:#4caf50; font-weight:700;" if r["best_strategy"] == "B" else "color:#888;"
            arb_html += f"""
                <tr style="border-bottom:1px solid #333;">
                    <td style="padding:4px 6px; color:#e0e0e0; max-width:160px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;" title="{r['event']}">{r['event'][:35]}</td>
                    <td style="padding:4px 6px; color:#e0e0e0; font-weight:500;" title="Poly: {r['outcome']}&#10;Kalshi: {r['outcome_kalshi']}">{r['outcome'][:30]}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; color:#e0e0e0;">{r['poly_yes']:.1%}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; color:#b0b0b0;">{r['poly_no']:.1%}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; color:#e0e0e0;">{r['kalshi_yes']:.1%}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; color:#b0b0b0;">{r['kalshi_no']:.1%}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; {cost_a_style}" title="Yes@Poly + No@Kalshi">${r['cost_a']:.2f}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; {cost_b_style}" title="Yes@Kalshi + No@Poly">${r['cost_b']:.2f}</td>
                    <td style="padding:4px 6px; text-align:right; font-family:monospace; color:#4caf50; font-weight:700;">{profit_pct:.1f}¢</td>
                    <td style="padding:4px 6px; color:#64b5f6; font-size:11px;">{r['direction']}</td>
                    <td style="padding:4px 6px; text-align:center;">{match_emoji}{r['outcome_sim']:.0%}{inv_badge}</td>
                </tr>
            """
        arb_html += """
            </table>
            <p style="margin:10px 0 0 0; color:#666; font-size:11px;">
                Cost A = Yes@Poly + No@Kalshi &nbsp;|&nbsp; Cost B = Yes@Kalshi + No@Poly
                &nbsp;|&nbsp; P.Yes/P.No = Polymarket &nbsp;|&nbsp; K.Yes/K.No = Kalshi
            </p>
        </div>"""
        display(HTML(arb_html))

    # ── All Spreads Table ──
    spread_html = f"""
    <div style="border:2px solid #2196f3; border-radius:12px; padding:20px; margin:16px 0;
                background:linear-gradient(135deg, #0d1b30 0%, #16213e 100%); color:#e0e0e0;
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <h2 style="margin:0 0 8px 0; color:#2196f3;">📈 All Cross-Platform Price Spreads</h2>
        <p style="margin:0 0 6px 0; color:#888; font-size:13px;">
            Top {min(TOP_ARBS, len(arb_rows))} of {len(arb_rows)} matched outcomes
            (spread ≥ ${MIN_SPREAD:.2f}).
            Shows all 4 prices + both strategy costs. Platform odds may not sum to 100%.
        </p>
        <table style="width:100%; border-collapse:collapse; font-size:11px;">
            <tr style="border-bottom:2px solid #2196f3;">
                <th style="text-align:left; padding:5px 4px; color:#90caf9;">Event</th>
                <th style="text-align:left; padding:5px 4px; color:#90caf9;">Poly Outcome</th>
                <th style="text-align:left; padding:5px 4px; color:#90caf9;">Kalshi Outcome</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">P.Yes</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">P.No</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">K.Yes</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">K.No</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">Y Sprd</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">N Sprd</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">Cost A</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">Cost B</th>
                <th style="text-align:right; padding:5px 4px; color:#90caf9;">Arb</th>
                <th style="text-align:center; padding:5px 4px; color:#90caf9;">Match</th>
            </tr>
    """
    for r in arb_rows[:TOP_ARBS]:
        arb_val = r["arb_profit"] * 100
        is_arb = r["arb_profit"] > 0
        arb_color = "#4caf50" if is_arb else "#666"
        arb_text = f"{arb_val:.1f}¢" if is_arb else "—"
        match_emoji = "🟢" if r["outcome_sim"] >= 0.85 else ("🟡" if r["outcome_sim"] >= 0.7 else "🟠")
        inv_badge = " 🔄" if r["inverted"] else ""

        # Highlight which cost is better
        cost_a_style = "color:#4caf50; font-weight:700;" if r["cost_a"] < r["cost_b"] else "color:#888;"
        cost_b_style = "color:#4caf50; font-weight:700;" if r["cost_b"] < r["cost_a"] else "color:#888;"
        if r["cost_a"] == r["cost_b"]:
            cost_a_style = cost_b_style = "color:#888;"

        spread_html += f"""
            <tr style="border-bottom:1px solid #222;">
                <td style="padding:3px 4px; color:#e0e0e0; max-width:130px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;" title="{r['event']}">{r['event'][:28]}</td>
                <td style="padding:3px 4px; color:#e0e0e0; font-weight:500;" title="{r['outcome']}">{r['outcome'][:25]}</td>
                <td style="padding:3px 4px; color:#b0b0b0;" title="{r['outcome_kalshi']}">{r['outcome_kalshi'][:25]}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:#e0e0e0;">{r['poly_yes']:.1%}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:#b0b0b0;">{r['poly_no']:.1%}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:#e0e0e0;">{r['kalshi_yes']:.1%}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:#b0b0b0;">{r['kalshi_no']:.1%}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:#ff9800;">{r['yes_spread']*100:.1f}¢</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:#ff9800;">{r['no_spread']*100:.1f}¢</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; {cost_a_style}" title="Y@P+N@K">${r['cost_a']:.2f}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; {cost_b_style}" title="Y@K+N@P">${r['cost_b']:.2f}</td>
                <td style="padding:3px 4px; text-align:right; font-family:monospace; color:{arb_color}; font-weight:{'700' if is_arb else '400'};">{arb_text}</td>
                <td style="padding:3px 4px; text-align:center;" title="Event: {r['event_sim']:.0%} Outcome: {r['outcome_sim']:.0%}">{match_emoji}{r['outcome_sim']:.0%}{inv_badge}</td>
            </tr>
        """
    spread_html += f"""
        </table>
        <p style="margin:10px 0 0 0; color:#666; font-size:11px;">
            Cost A = Yes@Poly + No@Kalshi &nbsp;|&nbsp; Cost B = Yes@Kalshi + No@Poly
            &nbsp;|&nbsp; Y Sprd = |P.Yes − K.Yes| &nbsp;|&nbsp; N Sprd = |P.No − K.No|
            &nbsp;|&nbsp; 🟢 ≥85%&ensp; 🟡 ≥70%&ensp; 🟠 ≥{MATCH_THRESHOLD:.0%}&ensp; 🔄 = negation-inverted
        </p>
    </div>"""
    display(HTML(spread_html))

    # ── Detailed Arbitrage Cards ──
    if true_arbs:
        display(HTML("<h2 style='color:#4caf50; border-bottom:2px solid #4caf50; padding-bottom:6px; margin-top:24px;'>🔍 Arbitrage Detail Cards</h2>"))
        for idx, r in enumerate(true_arbs[:10], 1):
            profit_pct = r["arb_profit"] * 100
            roi = (r["arb_profit"] / r["best_cost"] * 100) if r["best_cost"] > 0 else 0
            inv_note = "<span style='color:#ff9800;'>🔄 Negation-inverted match — Kalshi prices flipped for comparison</span><br>" if r["inverted"] else ""

            # Overround display
            poly_or_pct  = r["poly_overround"] * 100
            kalshi_or_pct = r["kalshi_overround"] * 100
            poly_or_color  = "#f44336" if r["poly_overround"] > 1.0 else "#4caf50"
            kalshi_or_color = "#f44336" if r["kalshi_overround"] > 1.0 else "#4caf50"

            card = f"""
            <div style="border:2px solid #4caf50; border-radius:12px; padding:18px; margin:12px 0;
                        background:#0d1b0d; color:#e0e0e0;
                        font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
                <h3 style="margin:0 0 6px 0; color:#4caf50;">#{idx} — {r['outcome']}</h3>
                <p style="margin:0 0 4px 0; color:#888; font-size:13px;">Event: <b>{r['event']}</b></p>
                <p style="margin:0 0 10px 0; color:#666; font-size:12px;">
                    Poly: "{r['outcome']}" ↔ Kalshi: "{r['outcome_kalshi']}"
                    &nbsp;<span style="color:#888;">(match {r['outcome_sim']:.0%})</span>
                </p>
                {inv_note}

                <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-bottom:14px;">
                    <div style="background:#0d1b30; padding:12px 14px; border-radius:8px; border:1px solid #2196f3;">
                        <p style="margin:0 0 8px 0; color:#2196f3; font-weight:600; font-size:13px;">POLYMARKET</p>
                        <div style="display:flex; justify-content:space-between;">
                            <div>
                                <p style="margin:0; color:#888; font-size:10px;">YES</p>
                                <p style="margin:2px 0 0 0; font-size:22px; font-weight:700; color:#e0e0e0; font-family:monospace;">{r['poly_yes']:.1%}</p>
                            </div>
                            <div>
                                <p style="margin:0; color:#888; font-size:10px;">NO</p>
                                <p style="margin:2px 0 0 0; font-size:22px; font-weight:700; color:#b0b0b0; font-family:monospace;">{r['poly_no']:.1%}</p>
                            </div>
                        </div>
                        <p style="margin:6px 0 0 0; font-size:11px;">
                            Overround: <span style="color:{poly_or_color};">{poly_or_pct:.1f}%</span>
                            &nbsp;|&nbsp; Liq: ${r['poly_liq']:,.0f}
                        </p>
                    </div>
                    <div style="background:#0d1b30; padding:12px 14px; border-radius:8px; border:1px solid #ff9800;">
                        <p style="margin:0 0 8px 0; color:#ff9800; font-weight:600; font-size:13px;">KALSHI{" (inverted)" if r['inverted'] else ""}</p>
                        <div style="display:flex; justify-content:space-between;">
                            <div>
                                <p style="margin:0; color:#888; font-size:10px;">YES</p>
                                <p style="margin:2px 0 0 0; font-size:22px; font-weight:700; color:#e0e0e0; font-family:monospace;">{r['kalshi_yes']:.1%}</p>
                            </div>
                            <div>
                                <p style="margin:0; color:#888; font-size:10px;">NO</p>
                                <p style="margin:2px 0 0 0; font-size:22px; font-weight:700; color:#b0b0b0; font-family:monospace;">{r['kalshi_no']:.1%}</p>
                            </div>
                        </div>
                        <p style="margin:6px 0 0 0; font-size:11px;">
                            Overround: <span style="color:{kalshi_or_color};">{kalshi_or_pct:.1f}%</span>
                            &nbsp;|&nbsp; Liq: ${r['kalshi_liq']:,.0f}
                        </p>
                    </div>
                </div>

                <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:10px; margin-bottom:14px;">
                    <div style="background:{'#1a2e1a' if r['best_strategy'] == 'A' else '#1a1a1a'}; padding:10px 14px; border-radius:8px;
                                border:2px solid {'#4caf50' if r['best_strategy'] == 'A' else '#333'};">
                        <p style="margin:0; color:#888; font-size:10px;">STRATEGY A: Y@Poly + N@Kalshi</p>
                        <p style="margin:4px 0 0 0; font-size:20px; font-weight:700; color:{'#4caf50' if r['profit_a'] > 0 else '#888'}; font-family:monospace;">
                            ${r['cost_a']:.4f}
                        </p>
                        <p style="margin:2px 0 0 0; color:#888; font-size:11px;">
                            Profit: {r['profit_a']*100:.1f}¢{"  ✓ BEST" if r['best_strategy'] == 'A' else ""}
                        </p>
                    </div>
                    <div style="background:{'#1a2e1a' if r['best_strategy'] == 'B' else '#1a1a1a'}; padding:10px 14px; border-radius:8px;
                                border:2px solid {'#4caf50' if r['best_strategy'] == 'B' else '#333'};">
                        <p style="margin:0; color:#888; font-size:10px;">STRATEGY B: Y@Kalshi + N@Poly</p>
                        <p style="margin:4px 0 0 0; font-size:20px; font-weight:700; color:{'#4caf50' if r['profit_b'] > 0 else '#888'}; font-family:monospace;">
                            ${r['cost_b']:.4f}
                        </p>
                        <p style="margin:2px 0 0 0; color:#888; font-size:11px;">
                            Profit: {r['profit_b']*100:.1f}¢{"  ✓ BEST" if r['best_strategy'] == 'B' else ""}
                        </p>
                    </div>
                    <div style="background:#2e1a00; padding:10px 14px; border-radius:8px; border:2px solid #ff9800;">
                        <p style="margin:0; color:#ff9800; font-size:10px;">SPREADS</p>
                        <p style="margin:4px 0 0 0; font-size:14px; font-weight:600; color:#ff9800; font-family:monospace;">
                            Yes: {r['yes_spread']*100:.1f}¢
                        </p>
                        <p style="margin:2px 0 0 0; font-size:14px; font-weight:600; color:#ff9800; font-family:monospace;">
                            No:&nbsp; {r['no_spread']*100:.1f}¢
                        </p>
                    </div>
                </div>

                <p style="margin:0; padding:8px 12px; background:#1a1a2e; border-radius:6px; border-left:4px solid #2196f3; font-size:13px; color:#64b5f6;">
                    <b>Action:</b> {r['direction']}
                    &nbsp;|&nbsp; ROI: {roi:.1f}%
                </p>
                <p style="margin:6px 0 0 0; font-size:11px; color:#555;">
                    Match quality: Event {r['event_sim']:.0%} • Outcome {r['outcome_sim']:.0%} &nbsp;|&nbsp;
                    {"<a href='" + r['poly_url'] + "' style='color:#64b5f6;'>Polymarket ↗</a>" if r['poly_url'] else ""}
                    {"&nbsp;&nbsp;<a href='" + r['kalshi_url'] + "' style='color:#64b5f6;'>Kalshi ↗</a>" if r['kalshi_url'] else ""}
                </p>
            </div>
            """
            display(HTML(card))
    else:
        display(HTML("""
        <div style="padding:20px; text-align:center; color:#ff9800; font-size:15px;
                    border:2px dashed #ff9800; border-radius:12px; margin:16px 0;
                    background:#1a1400;">
            No true arbitrage opportunities found at current thresholds.<br>
            Price spreads exist but combined cost ≥ $1.00 for all matched outcomes.<br>
            The spreads table above still shows pricing differences worth monitoring.
        </div>"""))